# Data Cleaning 

This notebook cleans the raw data available in data/raw and writes the clean version back to the folder data/processed. 

In [17]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from c08_farming_exit import config, features #access the config content via e.g. config.RAW_DATA_DIR
from c08_farming_exit.data_cleaning import load, most_common_or_nan

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [18]:
# #In case you want to run Stata in a cell using the magic command %%stata, initialize it first!
# from c08_farming_exit.stata_utils import init_stata
# init_stata()

## 1. Read out and merge raw data

In [19]:
COUNTRIES = {
    "Botswana": config.RAW_DATA_DIR / "Botswana",
    "Kenya":    config.RAW_DATA_DIR / "Kenya",
    "Namibia":  config.RAW_DATA_DIR / "Namibia",
    "Tanzania": config.RAW_DATA_DIR / "Tanzania",
    "Zambia":   config.RAW_DATA_DIR / "Zambia",
}

In [20]:
database = []

for country, base_path in COUNTRIES.items():
    identifying_info    = load(base_path, f"{country}_identifying_info.csv",                  features.IDENTIFYING_INFO_2023)
    hh_members          = load(base_path, f"{country}_household_members_characterstics.csv",  features.HH_MEMBERS_2023)
    on_farm_empl        = load(base_path, f"{country}_on_farm_employment.csv",                features.ON_FARM_EMPLOYMENT_2023)
    off_farm_empl       = load(base_path, f"{country}_off_farm_employment.csv",               features.OFF_FARM_EMPLOYMENT_2023)
    aspirations         = load(base_path, f"{country}_aspirations.csv",                       features.ASPIRATIONS_2023)
    child_aspirations   = load(base_path, f"{country}_child_aspiration.csv",                  features.CHILD_ASPIRATIONS_2023)
    land_ownership      = load(base_path, f"{country}_land_ownership_and_access.csv",         features.LAND_OWNERSHIP_ACCESS_2023)
    # livestock_ownership = load(base_path, f"{country}_livestock_ownership.csv",               features.LIVESTOCK_OWNERSHIP_2023) ## many datapoints per adult
    market_access       = load(base_path, f"{country}_market_access.csv",                     features.MARKET_ACCESS_2023)
    # shocks_and_coping   = load(base_path, f"{country}_shocks_and_coping.csv",                 features.SHOCKS_AND_COPING_2023) ## many datapoints per adult


    most_common_child_aspiration = (
        child_aspirations
        .groupby(["interview_key", "members_id"])['child_aspiration_continue_farming']
        .agg(most_common_or_nan)
        .reset_index()
        .rename(columns={'child_aspiration_continue_farming': 'most_common_child_aspiration'})
    )

    #MERGING ONLY DATASETS THAT ARE AVAILABLE
    merge = identifying_info.merge(hh_members, on=["interview_key"], how="inner")

    optional_merges = [
        (on_farm_empl,                 ["interview_key", "members_id"],  "left"),
        (off_farm_empl,                ["interview_key", "members_id"],  "left"),
        (aspirations,                  ["interview_key", "members_id"],  "left"),
        (most_common_child_aspiration, ["interview_key", "members_id"], "left"),
        (land_ownership,               ["interview_key"],                "left"),
        # (livestock_ownership,             ["interview_key"],                "left"),
        (market_access,                 ["interview_key"],                "left"),
        #(shocks_and_coping,                ["interview_key"],                "left"),
    ]

    for df, keys, how in optional_merges:
        if df is not None:
            merge = merge.merge(df, on=keys, how=how)

    merge['age'] = pd.to_numeric(merge['age'], errors='coerce')

    # FITERING: ONLY KEEPING ADULTS WITH FILLED TARGED VARIABLES
    filtered = merge[
        merge["relation_to_head"].isin(["Self/Head", 
                                             "Wife/Husband", 
                                             "Son/Daughter-In-Law", 
                                             "Sister/Brother", 
                                             "Mother/Father", 
                                             "Brother/Sister-In-Law", 
                                             "Grandfather/Mother", 
                                             "Father/Mother-In-Law"]) &
                                        (merge["age"] >= 18) &
                                        (merge["aspired_occupation_5_years_ahead"].notnull()) &
                                        (merge["aspiration_continue_farming"].notnull())
    ]

    database.append(filtered)

df = pd.concat(database, ignore_index=True)
df["personal_id"] = df["country"] + "_" + df["interview_key"].astype(str) + "_" + df["members_id"].astype(str)

[Botswana_identifying_info.csv] Missing columns: ['ea', 'region']
[Namibia_identifying_info.csv] Missing columns: ['dist']
[Tanzania_market_access.csv] File not found: C:\Users\localuser\my_projects\c08-farming-exit\data\raw\Tanzania\Tanzania_market_access.csv


## 2. Write clean data to data/raw folder

In [21]:
df.to_csv(config.PROCESSED_DATA_DIR / "clean_data.csv", index=False)